# PageInex -- Vectorless RAG

In [1]:
import os, json, time

from dotenv import load_dotenv
load_dotenv()

os.environ['PAGEINDEX_API_KEY'] = os.getenv('PAGEINDEX_API_KEY')
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')


In [5]:
from pageindex import PageIndexClient
from langchain.chat_models import init_chat_model

pi_key = os.environ['PAGEINDEX_API_KEY']
pi_client = PageIndexClient(api_key = pi_key)
llm = init_chat_model(model='groq:qwen/qwen32b-3b')


In [6]:
# Uploading pdf to PageIndex Cloud

PDF_PATH = '../RAGS/data/pdf/Yash Prasad 229309105 Major Project Report V2.pdf'
result = pi_client.submit_document(PDF_PATH)
doc_id = result['doc_id']

print(f'Uploaded {PDF_PATH}\nDocument ID: {doc_id}')

Uploaded ../RAGS/data/pdf/Yash Prasad 229309105 Major Project Report V2.pdf
Document ID: pi-cmrlvdmy500eo01o4z08rkscg


In [10]:
# tree index
while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get('status')

    if status == 'completed':
        print('Tree index is ready')
        break
    elif status == 'failed':
        print('\n Processing Failed. Check pdf format.')
        break
    time.sleep(5)

Tree index is ready


Tree Structure <br>
Each node has:
- node_id: unique id used during retrieval
- title: section heading
- page_inde: page number in original PDF
- text: section summary (when node_summary = True)
- nodes: child sections (nested)

In [11]:
# fetch tree
tree_result = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get('result', [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 1

🌲 Raw tree (first node):
{
  "title": "Healthcare Data Ingestion and Migration",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "This document comprises the title page and formal certification for a B.Tech project titled 'Healthcare Data Ingestion and Migration,' conducted by student Yash Prasad at Manipal University Jaipur in collaboration with Veersa Technologies. It includes academic and industry supervisor endorsements, departmental approval, and the project's timeline for the spring 2026 semester.",
  "text": "# Healthcare Data Ingestion and Migration\n\nA PROJECT REPORT\n\nSubmitted in partial fulfilment\nof the\nrequirement for the award of the degree of\n\nBACHELOR OF TECHNOLOGY (B. Tech)\n\nin\n\nComputer Science and Engineering (AIML)\n\nby\n\nYash Prasad\n\n229309105\n\nUnder the supervision of\n\nMr.Ajay Kumar Tyagi\nVeersa Technologies\n\n![img-0.jpeg](img-0.jpeg)\n\nMANIPAL UNIVERSITY\nJAIPUR\n\n(University under Section 2(f) of the U

In [14]:
# pretty-printing the full tree
def print_tree(nodes, indent=0):
    '''Recursively prints tree title'''
    for node in nodes:
        prefix = " "*indent + ('--' if indent > 0 else "")
        page = node.get('page_index', '?')
        print(f"{prefix}[{node['node_id']}] {node['title']} (p.{page})")
        if node.get('nodes'):
            print_tree(node['nodes'], indent + 1)

print('Full document structure: ')
print_tree(pageindex_tree)

Full document structure: 
[0000] Healthcare Data Ingestion and Migration (p.1)
 --[0001] ACKNOWLEDGMENTS (p.4)
 --[0002] LIST OF TABLES (p.6)
 --[0003] LIST OF FIGURES (p.7)
 --[0004] Contents (p.8)
 --[0005] 1 Introduction (p.9)
  --[0006] 1.1 Scope of the Work (p.10)
  --[0007] 1.2 Product Scenario (p.11)
 --[0008] 2 Requirement Analysis (p.13)
  --[0009] 2.1 Functional Requirements (p.13)
  --[0010] 2.2 Non-Functional Requirements (p.15)
  --[0011] 2.3 Use Case Scenario (p.16)
   --[0012] Use Case 1: Data Ingestion Workflow (p.16)
   --[0013] Use Case 2: Pipeline Extraction and Configuration Generation (p.16)
   --[0014] Use Case 3: Pipeline Deployment using Databricks Asset Bundles (p.17)
   --[0015] Use Case 4: Data Transformation Workflow (p.17)
   --[0016] Use Case 5: Data Validation and Reconciliation (p.17)
 --[0017] 3 System Design (p.20)
  --[0018] 3.1 Design Goals (p.20)
  --[0019] 3.2 System Architecture (p.21)
  --[0020] 3.3 Detailed Methodology (p.23)
 --[0021] 4 Work Do

In [15]:
# count total nodes
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get('nodes'):
            total += count_nodes(n['nodes'])
    return total

total = count_nodes(pageindex_tree)
print('Total nodes in tree: ',total)

Total nodes in tree:  27


# LLM Tree Search

pageindex_retrieval:
    query + tree -> LLM reasons -> "node 34 and 32 contain the answer"

In [44]:
from pydantic import BaseModel

class Output(BaseModel):
    thinking: str
    node_list: list


# llm Tree search function
def llm_tree_search(query: str, tree: list, model:str = 'groq:llama-3.1-8b-instant') -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids

    Returns: dict with 'thinking' and 'node_list'
    """

    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                'node_id': n['node_id'],
                'title': n['title'],
                'page': n.get('page_index', '?'),
                'summary': n.get('text', '')[:200]
            }
            if n.get('nodes'):
                entry['children'] = compress(n['nodes'])
            out.append(entry)
        return out

    compressed_tree = compress(tree)

    prompt=  f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""
    
    llm = init_chat_model(model=model)
    llm_structured_output = llm.with_structured_output(Output)
    response = llm_structured_output.invoke(prompt)
    return response

In [45]:
# sample query test
query = 'What is databricks?'
results = llm_tree_search(query, pageindex_tree)
# print(results)


print("🧠 LLM Reasoning:")
print(results.thinking)
print()
print("🎯 Selected Node IDs:", results.node_list)

🧠 LLM Reasoning:
Analyzing the query 'What is Databricks?' and the document tree, I identify relevant sections as '2 Requirement Analysis' and '3 System Design'. Within these sections, I consider nodes related to data processing, architecture, and deployment. Specifically, I focus on '2.3 Use Case Scenario' and '3.2 System Architecture'. Finally, I pinpoint 'Use Case 3: Pipeline Deployment using Databricks Asset Bundles' and '3.2 System Architecture' as the most relevant nodes.

🎯 Selected Node IDs: ['0011', '0014', '0019']


In [46]:
# sample query test
query = 'What are penguins?'
results = llm_tree_search(query, pageindex_tree)
# print(results)


print("🧠 LLM Reasoning:")
print(results.thinking)
print()
print("🎯 Selected Node IDs:", results.node_list)

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=Output> {"thinking": "Step 1: Analyze the query and identify relevant keywords. The query is \'What are penguins?\' but there is no mention of penguins in the given document tree. Looking at the document tree, I see that it is related to healthcare data ingestion and migration.", "node_list": ["0005", "0008"]}'}}